# 00 · Data setup and cohort validation

Loads the matched longitudinal PSM export and the full-cohort `PPMI_basic1.xlsx`, rescales time to months, and writes a derived RDS that later notebooks reuse.

In [1]:
source("helpers/pain_helpers.R")

df_long <- load_matched_long()
cat("rows:", nrow(df_long), "  cols:", ncol(df_long), "\n")
cat("unique PATNO:", dplyr::n_distinct(df_long$PATNO), "\n")
print(df_long %>% dplyr::distinct(PATNO, will_receive_dbs) %>% dplyr::count(will_receive_dbs, name = "n_patients"))
print(df_long %>% dplyr::distinct(PATNO, traj) %>% dplyr::count(traj, name = "n_patients"))

rows: 6320   cols: 106 


unique PATNO: 170 


# A tibble: 2 × 2
  will_receive_dbs n_patients
  <lgl>                 <int>
1 FALSE                   106
2 TRUE                     64


# A tibble: 3 × 2
  traj      n_patients
  <fct>          <int>
1 Pre-DBS           52
2 Post-DBS          64
3 Never-DBS        106


In [2]:
range(df_long$time_pos, na.rm = TRUE)
range(df_long$time_pos_months, na.rm = TRUE)
summary(df_long$NP1PAIN)
summary(df_long$weight_sw_trim90)

[1]    0 4021

[1]   0.0000 132.1068

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  0.000   0.000   1.000   1.101   2.000   4.000 

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.9386  0.9386  0.9789  1.0166  1.0594  1.2018 

In [3]:
pain_dedup <- dedup_earliest_per_bin(df_long)
cat("dedup rows:", nrow(pain_dedup), "  unique PATNO:", dplyr::n_distinct(pain_dedup$PATNO), "\n")
pain_dedup %>% dplyr::count(months) %>% dplyr::arrange(months)

dedup rows: 647   unique PATNO: 170 


months,n
<dbl>,<int>
-114,1
-102,1
-90,1
-78,1
-66,1
-54,14
-42,27
-30,43
-18,69


In [4]:
# Visits per patient (longitudinal balance check)
visits <- pain_dedup %>% dplyr::count(PATNO, will_receive_dbs, name = "n_visits")
visits %>% dplyr::group_by(will_receive_dbs) %>%
  dplyr::summarise(
    n_patients = dplyr::n(),
    mean_visits = mean(n_visits),
    median_visits = stats::median(n_visits),
    min_visits = min(n_visits),
    max_visits = max(n_visits),
    .groups = "drop"
  )

will_receive_dbs,n_patients,mean_visits,median_visits,min_visits,max_visits
<lgl>,<int>,<dbl>,<dbl>,<int>,<int>
FALSE,106,3.169811,3,1,8
TRUE,64,4.859375,5,2,9


In [5]:
# Save the derived long-format object for downstream notebooks
save_object(df_long, "pain_long")
save_object(pain_dedup, "pain_dedup")
cat("Saved: pain_long.rds and pain_dedup.rds to outputs/objects/\n")

Saved: pain_long.rds and pain_dedup.rds to outputs/objects/
